# Data Workflow & Acquisition

**Topics:** DS lifecycle • project structure • reproducibility • CSV/Excel acquisition


**How to use:** Run cells top-to-bottom. Exercises are marked ✅.

---
## Session plan (120 minutes)
1. (0–10) What a data workflow looks like (DS lifecycle)
2. (10–30) Project structure & organizing work
3. (30–55) Reproducibility essentials (environment, seeds, logging, configs)
4. (55–95) Data acquisition from CSV/Excel (I/O, types, parsing, validation)
5. (95–115) Mini end-to-end workflow: raw → clean → output + report-ready table
6. (115–120) Wrap-up checklist


## 0. Setup
We’ll use **pandas** for reading CSV/Excel and a few standard libraries.

> Tip: If `openpyxl` is missing, install it with `pip install openpyxl`.


In [ ]:
import os
import sys
import platform
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd

print('Python:', sys.version.split()[0])
print('Platform:', platform.platform())
print('pandas:', pd.__version__)


## 1. The DS lifecycle (workflow map)
A typical (iterative) lifecycle:
1. **Problem framing**: goal, scope, success metric
2. **Data acquisition**: sources, permissions, extraction
3. **Data understanding**: schema, missingness, anomalies
4. **Data cleaning & feature engineering**
5. **Modeling / analysis**
6. **Evaluation**: metrics, validation, error analysis
7. **Communication & deployment**: dashboards, reports, pipelines
8. **Monitoring & maintenance**

**Key idea:** you don’t “finish” steps once; you loop.

✅ **Exercise (2 min):** Write a 1–2 sentence problem statement for a dataset you know (or make one up). Include a measurable outcome.


In [ ]:
# ✅ Exercise: Replace the text below with your own problem statement.


## 2. Project structure (repeatable and collaborative)
A clean structure reduces chaos and makes work reproducible.

### Recommended layout
```
project/
  README.md
  data/
    raw/        # immutable source files
    interim/    # intermediate outputs
    processed/  # analysis-ready
  notebooks/    # exploration, teaching
  src/          # reusable functions/modules
  reports/      # figures, tables, exports
  configs/      # settings, paths
  tests/        # unit/data tests
```

**Rule of thumb:** keep `raw/` read-only; never edit raw data in place.

### This notebook will create a tiny demo project folder
We’ll generate small CSV and Excel files so the notebook is self-contained.


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path.cwd() / "demo_project"
DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
REPORTS = PROJECT_ROOT / "reports"
CONFIGS = PROJECT_ROOT / "configs"

for p in [DATA_RAW, DATA_INTERIM, DATA_PROCESSED, REPORTS, CONFIGS]:
    p.mkdir(parents=True, exist_ok=True)

PROJECT_ROOT


### Add a simple README
A good README explains purpose, how to run, and where outputs go.


In [ ]:
readme_text = """# Demo Project: Data Workflow & Acquisition

## Purpose
Teaching example for:
- DS lifecycle
- Project structure
- Reproducibility basics
- Reading CSV/Excel reliably

## How to run
1. Create environment (optional): `python -m venv .venv` then install requirements
2. Run `notebooks/` in order

## Data folders
- `data/raw/`: immutable source files
- `data/interim/`: intermediate outputs
- `data/processed/`: analysis-ready outputs

## Outputs
- `reports/`: tables/figures for sharing
"""

(PROJECT_ROOT / "README.md").write_text(readme_text)
print("Wrote:", PROJECT_ROOT / "README.md")


## 3. Reproducibility essentials
Reproducibility means someone else (or future you) can rerun the work and get the same result.

### 3.1 Track your environment
- Pin package versions (`requirements.txt` / `pyproject.toml`)
- Record Python version

### 3.2 Control randomness
- Set seeds for `random` and `numpy`
- Log key parameters

### 3.3 Separate configuration from code
- Put paths/parameters in a JSON/YAML config

### 3.4 Make outputs deterministic
- Sort rows before saving
- Use explicit dtypes
- Avoid locale-dependent parsing


In [ ]:
# 3.1 Freeze a minimal requirements file (teaching demo)
requirements = [
    f"pandas=={pd.__version__}",
    f"numpy=={np.__version__}",
    "openpyxl"  # needed for reading/writing .xlsx with pandas
]
(PROJECT_ROOT / "requirements.txt").write_text("\n".join(requirements) + "\n")
print((PROJECT_ROOT / "requirements.txt").read_text())


In [ ]:
# 3.2 Control randomness
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

# Demonstrate determinism
np.random.rand(3)


### 3.3 Configuration file
We’ll store dataset filenames and some parsing options in `configs/config.json`.


In [ ]:
config = {
    "seed": SEED,
    "raw_csv": "sales_raw.csv",
    "raw_excel": "sales_raw.xlsx",
    "date_format": "%Y-%m-%d",
    "currency_columns": ["unit_price"],
}

config_path = CONFIGS / "config.json"
config_path.write_text(json.dumps(config, indent=2))
print("Wrote:", config_path)
print(config_path.read_text())


## 4. Create demo data (CSV + Excel)
We’ll simulate messy real-world data: mixed date formats, missing values, and numeric columns stored as strings.


In [ ]:
# Generate a small "sales" dataset with intentional issues
n = 30
rng = np.random.default_rng(SEED)

df_raw = pd.DataFrame({
    "order_id": range(1001, 1001+n),
    "order_date": pd.to_datetime("2025-01-01") + pd.to_timedelta(rng.integers(0, 60, size=n), unit="D"),
    "region": rng.choice(["North", "South", "East", "West"], size=n),
    "product": rng.choice(["A", "B", "C"], size=n),
    "units": rng.integers(1, 12, size=n),
    "unit_price": np.round(rng.uniform(5, 30, size=n), 2),
})

# Introduce messy formats
# - some dates as strings in different formats
# - some unit_price as strings with currency symbol
# - some missing values
messy = df_raw.copy()

# date strings
for i in [2, 5, 11]:
    messy.loc[i, "order_date"] = messy.loc[i, "order_date"].strftime("%m/%d/%Y")
for i in [7, 13]:
    messy.loc[i, "order_date"] = messy.loc[i, "order_date"].strftime("%d-%b-%Y")

# currency strings
for i in [3, 9, 18]:
    messy.loc[i, "unit_price"] = f"${messy.loc[i, 'unit_price']}"

# missing values
messy.loc[4, "region"] = None
messy.loc[15, "units"] = None

messy.head(10)


In [ ]:
# Save to CSV and Excel (raw)
raw_csv_path = DATA_RAW / config["raw_csv"]
raw_xlsx_path = DATA_RAW / config["raw_excel"]

messy.to_csv(raw_csv_path, index=False)

with pd.ExcelWriter(raw_xlsx_path, engine="openpyxl") as writer:
    messy.to_excel(writer, index=False, sheet_name="sales")

raw_csv_path, raw_xlsx_path


## 5. Data acquisition from CSV
### 5.1 Basic read
Start with a straightforward read, then inspect.


In [ ]:
df_csv = pd.read_csv(raw_csv_path)
print(df_csv.shape)
df_csv.head()


### 5.2 Inspect schema and quality quickly
Always check:
- column names
- dtypes
- missing values
- duplicates


In [ ]:
df_csv.info()

In [ ]:
df_csv.isna().sum().sort_values(ascending=False)

In [ ]:
df_csv.duplicated().sum()

### 5.3 Robust parsing: dtypes + dates
Common pitfalls:
- `order_date` may not parse automatically
- `unit_price` contains `$` so becomes object (string)
- `units` has missing values so may become float

We'll:
1. Parse dates with `to_datetime(errors='coerce')`
2. Clean currency symbols and cast to numeric
3. Convert `units` to nullable integer (`Int64`)


In [ ]:
df = df_csv.copy()

# Parse dates (handles multiple formats reasonably well)
df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce", infer_datetime_format=True)

# Clean currency
(df["unit_price"].astype(str)
   .str.replace("$", "", regex=False)
   .astype(float)
)

df["unit_price"] = (
    df["unit_price"].astype(str)
      .str.replace("$", "", regex=False)
)
df["unit_price"] = pd.to_numeric(df["unit_price"], errors="coerce")

# Nullable integer for units
# (allows missing values without converting the whole column to float)
df["units"] = pd.to_numeric(df["units"], errors="coerce").astype("Int64")

# Derived column
df["revenue"] = df["units"].astype("float") * df["unit_price"]

df.info()


✅ **Exercise (5 min):** Add a validation check that `unit_price` is always positive and `units` is >= 1 when present. Print any rows that violate.


In [ ]:
# ✅ Exercise: implement validation checks

### 5.4 Save an intermediate cleaned dataset (deterministic)
Best practices:
- sort by a stable key
- save without index
- use a processed format (CSV/Parquet)


In [ ]:
df_sorted = df.sort_values(["order_date", "order_id"]).reset_index(drop=True)

interim_path = DATA_INTERIM / "sales_interim.csv"
df_sorted.to_csv(interim_path, index=False)

print("Wrote:", interim_path)
df_sorted.head()


## 6. Data acquisition from Excel
Excel pitfalls:
- multiple sheets
- mixed types per column
- empty rows/columns

### 6.1 List sheets and read a specific sheet


In [ ]:
xlsx = pd.ExcelFile(raw_xlsx_path)
xlsx.sheet_names


In [ ]:
df_xlsx = pd.read_excel(raw_xlsx_path, sheet_name="sales")
print(df_xlsx.shape)
df_xlsx.head()


### 6.2 Apply the same cleaning logic
Ideally, put this in a reusable function in `src/`.


In [ ]:
def clean_sales(df_in: pd.DataFrame) -> pd.DataFrame:
    df = df_in.copy()
    df.columns = [c.strip().lower() for c in df.columns]

    df["order_date"] = pd.to_datetime(df["order_date"], errors="coerce", infer_datetime_format=True)

    df["unit_price"] = (
        df["unit_price"].astype(str)
          .str.replace("$", "", regex=False)
    )
    df["unit_price"] = pd.to_numeric(df["unit_price"], errors="coerce")

    df["units"] = pd.to_numeric(df["units"], errors="coerce").astype("Int64")
    df["revenue"] = df["units"].astype(float) * df["unit_price"]

    # Optional: enforce categories
    for c in ["region", "product"]:
        if c in df.columns:
            df[c] = df[c].astype("string")

    return df

cleaned_xlsx = clean_sales(df_xlsx)
cleaned_xlsx.info()


✅ **Exercise (5 min):** Modify `clean_sales` to drop rows where `order_id` is missing OR `order_date` failed parsing (is NaT).


In [ ]:
# ✅ Exercise solution (edit the function above if you want; here we apply a post-filter)

## 6B. Data acquisition from other sources (how it *usually* happens)

So far we loaded **files** (CSV/Excel). In real projects you often pull data from **multiple sources**:

- **APIs (REST/GraphQL):** call endpoints, paginate, handle auth + rate limits, store raw JSON.
- **Databases / warehouses:** SQL query (Postgres, Snowflake, BigQuery), extract to a staging area.
- **Cloud object storage:** read from S3/GCS/Azure Blob (often Parquet/CSV), versioned paths by date.
- **Web pages / scraping:** parse HTML (only where allowed), prefer official APIs when available.
- **Internal logs / product events:** clickstreams, app logs, telemetry; often semi-structured.
- **Streaming:** Kafka/Kinesis/PubSub for near-real-time ingestion into a lake/warehouse.
- **Manual / surveys:** spreadsheets, forms; validate aggressively (humans create surprises).

### A practical pattern

1. **Acquire raw** (exactly as received) → save to `data/raw/` with a timestamp or partition (e.g., `dt=2026-02-14/`).
2. **Normalize** into a tabular form (DataFrame) → apply the *same* cleaning/validation functions.
3. **Store intermediates** (`data/interim/`) and **curated outputs** (`data/processed/`) in a stable format (often Parquet).
4. **Log metadata**: source, pull time, parameters, schema, row counts, checks.


### 6B.1 API acquisition (REST) — structure + pagination + JSON normalization

We can't call the internet from this notebook, but this cell shows the **shape** of a typical API pull.
We also create a **local mock API response** (JSON) and normalize it into a DataFrame.


In [ ]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
DATA_DIR = PROJECT_DIR / "data"
RAW_DIR = DATA_DIR / "raw"
INTERIM_DIR = DATA_DIR / "interim"
PROCESSED_DIR = DATA_DIR / "processed"

for d in (RAW_DIR, INTERIM_DIR, PROCESSED_DIR):
    d.mkdir(parents=True, exist_ok=True)

RAW_DIR


In [ ]:
import json
from pathlib import Path

# --- Typical API pull skeleton (example only; requires internet/auth) ---
# import requests
# url = "https://api.example.com/v1/orders"
# headers = {"Authorization": f"Bearer {TOKEN}"}
# params = {"start_date": "2026-01-01", "page": 1}
# all_rows = []
# while True:
#     r = requests.get(url, headers=headers, params=params, timeout=30)
#     r.raise_for_status()
#     payload = r.json()
#     all_rows.extend(payload["data"])
#     if payload.get("next_page") is None:
#         break
#     params["page"] = payload["next_page"]

# --- Local mock response (offline-friendly) ---
mock = {
    "data": [
        {"order_id": 2001, "customer": {"id": "C10", "segment": "SMB"}, "amount": 120.5, "order_date": "2026-02-01"},
        {"order_id": 2002, "customer": {"id": "C11", "segment": "ENT"}, "amount": 899.0, "order_date": "2026-02-03"},
        {"order_id": 2003, "customer": {"id": "C10", "segment": "SMB"}, "amount": 42.0,  "order_date": "2026-02-04"},
    ],
    "meta": {"pulled_at": "2026-02-14T10:00:00Z", "source": "mock_api", "version": "v1"}
}

raw_api_path = RAW_DIR / "orders_api_mock.json"
raw_api_path.write_text(json.dumps(mock, indent=2), encoding="utf-8")

# Normalize nested JSON to a flat table
df_api = pd.json_normalize(mock["data"])
df_api["customer_id"] = df_api["customer.id"]  # common step: make join keys explicit
df_api["order_date"] = pd.to_datetime(df_api["order_date"], errors="coerce")

df_api


### 6B.2 Database acquisition (SQL) — extract via query, keep it reproducible

A reproducible database pull includes:
- the **exact SQL** (checked into git),
- **parameter values** (dates, filters),
- and a **snapshot** of the extracted result stored in `data/raw/` (or a query-id in a warehouse).

Below we demonstrate using **SQLite locally** (same pattern applies to Postgres/Snowflake/BigQuery).


In [ ]:
import sqlite3
import pandas as pd

# Create a tiny SQLite DB in data/raw (for demo)
db_path = RAW_DIR / "demo_orders.sqlite"
conn = sqlite3.connect(db_path)

# Use our CSV-loaded data if it exists; otherwise, use the API mock table
try:
    source_df = df_csv.copy()
except NameError:
    source_df = df_api.copy()

# --- Make the dataset consistent across sources ---

# 1) Ensure numeric types (CSV/Excel often load numbers as strings like "1,000" or "$12.50")
def _to_number(s: pd.Series) -> pd.Series:
    # keep digits, minus sign, decimal point
    cleaned = s.astype(str).str.replace(r"[^0-9\.-]", "", regex=True)
    return pd.to_numeric(cleaned, errors="coerce")

for col in ["units", "unit_price", "amount"]:
    if col in source_df.columns:
        source_df[col] = _to_number(source_df[col])

# 2) Ensure we have an "amount" column for aggregation
if "amount" not in source_df.columns:
    if {"units", "unit_price"}.issubset(source_df.columns):
        source_df = source_df.assign(amount=source_df["units"] * source_df["unit_price"])
    else:
        raise ValueError(
            "No 'amount' column found and cannot derive it (need 'units' and 'unit_price'). "
            f"Columns: {list(source_df.columns)}"
        )

# 3) Basic data validation: drop rows where amount couldn't be parsed/derived
if source_df["amount"].isna().any():
    source_df = source_df.dropna(subset=["amount"]).copy()

# 4) Choose a grouping dimension that exists in the current source
group_candidates = ["customer_id", "region", "product"]
group_col = next((c for c in group_candidates if c in source_df.columns), None)

if group_col is None:
    raise ValueError(
        f"Couldn't find any grouping column among {group_candidates}. "
        f"Columns: {list(source_df.columns)}"
    )

# Optional: if API data has nested customer.id but no customer_id, create it
if "customer_id" not in source_df.columns and "customer.id" in source_df.columns:
    source_df = source_df.assign(customer_id=source_df["customer.id"])
    if group_col == "customer_id":
        group_col = "customer_id"

# Write to SQLite
source_df.to_sql("orders_raw", conn, if_exists="replace", index=False)

# Aggregate in SQL
sql = f"""
SELECT
  {group_col} AS group_key,
  COUNT(*) AS n_orders,
  ROUND(SUM(amount), 2) AS total_amount
FROM orders_raw
GROUP BY {group_col}
ORDER BY total_amount DESC
"""

df_sql = pd.read_sql_query(sql, conn)
conn.close()

df_sql

### 6B.3 Cloud object storage (S3/GCS/Azure) — common in production

In production you often pull from an object store (Parquet is common for analytics):

```python
# AWS S3 example
import pandas as pd
import s3fs

fs = s3fs.S3FileSystem(anon=False)
path = "s3://my-bucket/sales/dt=2026-02-14/sales.parquet"
df = pd.read_parquet(path, filesystem=fs)
```

Key practices:
- **Partitioning** by date (`dt=YYYY-MM-DD`) to make reads fast.
- **Immutable raw snapshots** (never overwrite yesterday's raw pull).
- **Schema evolution** strategy (new columns, type changes).


### 6B.4 Web pages / HTML parsing (when allowed)

Prefer official APIs. If you must parse HTML:
- confirm it's permitted (robots/ToS),
- cache the raw HTML you used,
- write tests for selectors (pages change).

Offline demo: parse a small HTML snippet locally.


In [ ]:
from bs4 import BeautifulSoup

html = '''
<table>
  <tr><th>product</th><th>price</th></tr>
  <tr><td>Widget</td><td>19.99</td></tr>
  <tr><td>Gadget</td><td>29.50</td></tr>
</table>
'''

soup = BeautifulSoup(html, "html.parser")
rows = []
for tr in soup.select("tr")[1:]:
    tds = [td.get_text(strip=True) for td in tr.select("td")]
    rows.append({"product": tds[0], "price": float(tds[1])})

df_html = pd.DataFrame(rows)
df_html


## 7. Data dictionary & documentation
A **data dictionary** reduces confusion about meanings and units.

We’ll create a simple dictionary as a DataFrame and export it.


In [ ]:
data_dict = pd.DataFrame([
    {"column": "order_id", "type": "int", "description": "Unique order identifier"},
    {"column": "order_date", "type": "date", "description": "Date the order was placed"},
    {"column": "region", "type": "string", "description": "Sales region (North/South/East/West)"},
    {"column": "product", "type": "string", "description": "Product code"},
    {"column": "units", "type": "int (nullable)", "description": "Number of units sold"},
    {"column": "unit_price", "type": "float", "description": "Price per unit (USD)"},
    {"column": "revenue", "type": "float", "description": "units * unit_price"},
])

dict_path = REPORTS / "data_dictionary.csv"
data_dict.to_csv(dict_path, index=False)

data_dict


## 8. Mini end-to-end workflow: raw → processed → report table
We’ll produce a clean, analysis-ready table and a simple summary for reporting.


In [ ]:
processed_path = DATA_PROCESSED / "sales_processed.csv"
cleaned = clean_sales(pd.read_csv(raw_csv_path))
cleaned = cleaned.dropna(subset=["order_id", "order_date"]).sort_values(["order_date", "order_id"])

cleaned.to_csv(processed_path, index=False)
print("Wrote:", processed_path)

cleaned.head()


In [ ]:
# Simple report-ready table: revenue by region and product
summary = (cleaned
           .groupby(["region", "product"], dropna=False)["revenue"]
           .sum(min_count=1)
           .reset_index()
           .sort_values("revenue", ascending=False)
          )

summary_path = REPORTS / "revenue_by_region_product.csv"
summary.to_csv(summary_path, index=False)
summary


## 9. Reproducibility checklist (take-home)
- [ ] Define the problem + success metric
- [ ] Use a consistent project structure
- [ ] Keep `data/raw` immutable
- [ ] Capture environment (`requirements.txt`)
- [ ] Set seeds; keep configs in files
- [ ] Validate schema and data quality
- [ ] Save deterministic outputs (sorted, typed)
- [ ] Document with a data dictionary

✅ **Optional extension:** Add unit tests for `clean_sales` (e.g., using `pytest`).
